# Prepare Maps

In this notebook we prepare observations to be used in the map_maker stage, as well as register what observations have existing map files.

## Setup Observations

#### Check available obs 🔍

We get the list of all observations that fit our search criteria.

In [ ]:
from sotodlib.core import Context, AxisManager, OffsetAxis
from sotodlib.tod_ops import flags, detrend_tod, jumps, gapfill, pca, filters, apodize, fft_ops
from sotodlib.core.flagman import has_any_cuts, has_all_cut
import numpy as np
import os, sys, subprocess
from astropy.time import Time
from astropy.coordinates import EarthLocation, AltAz, get_sun
from datetime import datetime
from zoneinfo import ZoneInfo
from datetime import datetime, timedelta
from sotodlib.io import hkdb
import astropy.units as u

In [3]:
import warnings
warnings.filterwarnings(action='ignore')

from sotodlib import core
import map_fun as mf
import msgspec

platform = "lat"
site = "nersc"
base_path = f"/global/cfs/cdirs/sobs/metadata/{platform}/"

align1 = 1744848000 # First Alignment
align2 = 1756684800 # Second Alignment
stop_time = 2748476800 # Corresponds to Febuary of year 2057

# Observations with the best pointing model available so far
context_path = '/global/cfs/cdirs/sobs/users/skh/data/beams/lat/context_nersc_pointing_model.yaml'
context = core.Context(context_path)
obslist_align1 = context.obsdb.query(f'mars and tube_slot=="i1" and type=="obs" and subtype=="cal" and start_time > {align1} and stop_time < {align2}', tags=['mars'])
obslist_align2 = context.obsdb.query(f'mars and tube_slot=="i1" and type=="obs" and subtype=="cal" and start_time > {align2} and stop_time < {stop_time}', tags=['mars'])
obslist = context.obsdb.query(f'mars and tube_slot=="i1" and type=="obs" and subtype=="cal" and start_time > {align1} and stop_time < {stop_time}', tags=['mars'])

print(f"{len(obslist_align1)} observations found for first alignment")
print(f"{len(obslist_align2)} observations found for second alignment")

154 observations found for first alignment
78 observations found for second alignment


#### Create Empty Map Dictionary 📚

We create an empty json file that will contain the list of all obs id's, alongside placeholder entries that we will fill throughout this notebook.

In [4]:
mf.maps_maps_path = "/global/u2/a/andrs/Products/Mars/i1/"
mf.maps_dict = "/global/u2/a/andrs/Products/Mars/i1/maps.json"

maps_meta = {}
from itertools import product

wafer_options = ["ws0", "ws1", "ws2"]
band_options = ["f090", "f150"]
placeholder_dict = {}

keys_dict = {
    "snr": 0,
    "fits": {},
    "fit_mesg": ''
}

for case in product(wafer_options, band_options):
    wafer, band = case    
    placeholder_dict.update({f"{band}-{wafer}": {
        "full": keys_dict,
        "lscans": keys_dict,
        "rscans": keys_dict
    }})

for obs_id in obslist['obs_id']:
    maps_meta[str(obs_id)] = placeholder_dict

encoded = msgspec.json.encode(maps_meta)

with open(mf.maps_dict, "wb") as f:
    f.write(encoded)

#### Update Times Metadata ⏳

For every observation we calculate the observation time, the sunrise/sunset, and classify that observation as `day`, `night` or `transition`.

In [6]:
with open(mf.maps_dict, "rb") as f:
    encoded = f.read()

maps_meta = msgspec.json.decode(encoded)

obs_ids = list(maps_meta.keys())

for obs_id in obs_ids:
    ctime = obslist[int(np.where(obslist['obs_id'] == obs_id)[0][0])]['timestamp']
    rise, set_, dt = get_obs_suntimes(ctime)
    time_class = classify_obs_time(rise, set_, dt)
    maps_meta[obs_id]['time_info'] = {}
    maps_meta[obs_id]['time_info']['time_class'] = time_class
    maps_meta[obs_id]['time_info']['sunrise'] = rise.strftime("%Y-%m-%d %H:%M:%S")
    maps_meta[obs_id]['time_info']['sunset'] = set_.strftime("%Y-%m-%d %H:%M:%S")
    maps_meta[obs_id]['time_info']['date'] = dt.strftime("%Y-%m-%d %H:%M:%S")
    maps_meta[obs_id]['time_info']['season_class'] = "first_alignment" if ctime < align2 else "second_alignment"

In [7]:
maps_meta = msgspec.json.encode(maps_meta)
with open(mf.maps_dict, "wb") as f:
    f.write(maps_meta)

## Make & Load Maps

#### Make Maps

This part is quite resource intensive, so this notebook assumes the maps already exist and were created using the appropriate mapping script.

#### Check Successfull Maps ✅

We take a look at the saved maps directory and check which observations have solved maps, thus indicating they were successfully mapped.

In [10]:
import json, glob, os, msgspec

with open(mf.maps_dict, "rb") as f:
    encoded = f.read()

maps_meta = msgspec.json.decode(encoded)
obslist = list(maps_meta.keys())

meta = maps_meta.copy()
map_counter = 0
obs_counter = 0

for obs in obslist:
    has_valid_map = False
    for key in list(maps_meta[obs].keys()):
        if key != "time_info":
            band, wafer = key.split("-")
            
            if os.path.isfile(f"/global/u2/a/andrs/Products/Mars/i1/{obs}/{obs}_{band}_{wafer}_solved.fits"):
                meta[obs][key]["nomap"] = False
                map_counter += 1
                has_valid_map = True
            else:
                meta[obs][key]["nomap"] = True
    
    if has_valid_map:
        obs_counter += 1

print(f"There are {map_counter} maps w/o counting scan splits. These maps originate from {obs_counter} observations.")

encoded = msgspec.json.encode(maps_meta)

with open(mf.maps_dict, "wb") as f:
    f.write(encoded)

There are 324 maps w/o counting scan splits. These maps originate from 70 observations.
